# Debug Hybrid Clustering Algorithms

This notebook helps diagnose why hybrid HDBSCAN algorithms produce unexpected results.

## Goals
1. Understand cluster statistics (d3, median, IQR, thresholds)
2. Debug specific merge/split decisions
3. Compare current implementation with original idea (median + k×IQR)
4. Test simplified algorithms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sim_bench.clustering.hybrid_hdbscan_knn import HybridHDBSCANKNN
from sim_bench.clustering.hybrid_closest_face import HybridHDBSCANClosestFace
from sim_bench.clustering.distance_utils import cosine_distance_matrix, cosine_distance_to_set

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Your Face Embeddings

Load the embeddings from your pipeline run. Adjust the path to your data.

In [ ]:
# Option 1: Load from cache or pipeline result
# Replace with your actual data loading code

# Example: Load from saved numpy array
# embeddings = np.load('path/to/embeddings.npy')
# ground_truth_labels = np.load('path/to/labels.npy')  # If you have ground truth

# For testing, create synthetic data
from sklearn.datasets import make_blobs

# Create synthetic embeddings with clear clusters
np.random.seed(42)
embeddings, true_labels = make_blobs(
    n_samples=200,
    n_features=512,
    centers=8,
    cluster_std=0.15,
    random_state=42
)

# Normalize for cosine distance
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print(f"Loaded {len(embeddings)} embeddings with shape {embeddings.shape}")

## 2. Run Clustering with Debug Data Collection

In [ ]:
# Choose algorithm to debug
algorithm = 'hybrid_hdbscan_knn'  # or 'hybrid_closest_face'

config = {
    'algorithm': algorithm,
    'min_cluster_size': 2,
    'min_samples': 2,
    'cluster_selection_epsilon': 0.045,
    'knn_k': 3,
    'threshold_percentile': 90,
    'threshold_floor': 0.125 if algorithm == 'hybrid_hdbscan_knn' else 0.045,
    'threshold_ceiling': 0.405,
    'merge_min_pairs': 3,
    'merge_min_distinct': 2,
    'attach_min_exemplars': 2,
    'split_enabled': True,
    'split_threshold': 0.65,
}

if algorithm == 'hybrid_hdbscan_knn':
    clusterer = HybridHDBSCANKNN(config)
else:
    clusterer = HybridHDBSCANClosestFace(config)

# Run with debug data collection
labels, stats = clusterer.cluster(embeddings, collect_debug_data=True)

print(f"\nClustering Results:")
print(f"  Clusters: {stats['n_clusters']}")
print(f"  Noise points: {stats['n_noise']}")
print(f"  Total merges: {stats.get('total_merges', 0)}")
print(f"  Total attached: {stats.get('total_attached', 0)}")
print(f"  Total splits: {stats.get('total_splits', 0)}")
print(f"\nCluster sizes: {stats['cluster_sizes']}")

## 3. Analyze Cluster Statistics

Compute d3 values and statistics for each cluster to understand thresholds.

In [ ]:
def compute_cluster_statistics(labels, embeddings, k=3):
    """Compute detailed statistics for each cluster."""
    cluster_stats = []
    
    for cluster_id in sorted(set(labels)):
        if cluster_id == -1:
            continue
            
        mask = labels == cluster_id
        indices = np.where(mask)[0]
        cluster_emb = embeddings[indices]
        n_faces = len(indices)
        
        if n_faces < 2:
            continue
        
        # Compute pairwise distances
        dist_matrix = cosine_distance_matrix(cluster_emb)
        
        # Compute d3 values (distance to k-th nearest neighbor)
        k_actual = min(k, n_faces - 1)
        d3_values = []
        for i in range(n_faces):
            sorted_dists = np.sort(dist_matrix[i])[1:k_actual + 1]  # Exclude self
            d3 = sorted_dists[-1] if len(sorted_dists) > 0 else 0
            d3_values.append(d3)
        d3_values = np.array(d3_values)
        
        # Compute all pairwise distances (excluding diagonal)
        upper_tri = dist_matrix[np.triu_indices_from(dist_matrix, k=1)]
        
        # Statistics
        d3_median = np.median(d3_values)
        d3_q1, d3_q3 = np.percentile(d3_values, [25, 75])
        d3_iqr = d3_q3 - d3_q1
        d3_p90 = np.percentile(d3_values, 90)
        
        pairwise_median = np.median(upper_tri)
        pairwise_q1, pairwise_q3 = np.percentile(upper_tri, [25, 75])
        pairwise_iqr = pairwise_q3 - pairwise_q1
        pairwise_p90 = np.percentile(upper_tri, 90)
        
        # Original idea thresholds
        t_accept_d3 = d3_median + 1.5 * d3_iqr
        t_split_d3 = d3_median + 2.5 * d3_iqr
        
        cluster_stats.append({
            'cluster_id': cluster_id,
            'n_faces': n_faces,
            # d3 statistics
            'd3_median': d3_median,
            'd3_q1': d3_q1,
            'd3_q3': d3_q3,
            'd3_iqr': d3_iqr,
            'd3_p90': d3_p90,
            'd3_min': np.min(d3_values),
            'd3_max': np.max(d3_values),
            # Pairwise statistics
            'pairwise_median': pairwise_median,
            'pairwise_q1': pairwise_q1,
            'pairwise_q3': pairwise_q3,
            'pairwise_iqr': pairwise_iqr,
            'pairwise_p90': pairwise_p90,
            'pairwise_min': np.min(upper_tri),
            'pairwise_max': np.max(upper_tri),
            # Original idea thresholds
            't_accept_d3': t_accept_d3,
            't_split_d3': t_split_d3,
            # Store values for plotting
            'd3_values': d3_values,
            'pairwise_values': upper_tri,
        })
    
    return pd.DataFrame(cluster_stats)

cluster_stats_df = compute_cluster_statistics(labels, embeddings, k=config['knn_k'])

# Display statistics
display_cols = ['cluster_id', 'n_faces', 'd3_median', 'd3_iqr', 'd3_p90', 
                't_accept_d3', 't_split_d3', 'pairwise_median', 'pairwise_p90']
print("\nCluster Statistics:")
print(cluster_stats_df[display_cols].round(4))

## 4. Compare Thresholds: Current vs Original Idea

In [ ]:
# Get actual thresholds used by algorithm
if 'debug' in stats and 'cluster_thresholds' in stats['debug']:
    actual_thresholds = stats['debug']['cluster_thresholds']
    cluster_stats_df['actual_threshold'] = cluster_stats_df['cluster_id'].map(actual_thresholds)
    
    # Compare thresholds
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Actual threshold vs d3-based threshold
    ax = axes[0]
    ax.scatter(cluster_stats_df['t_accept_d3'], cluster_stats_df['actual_threshold'], 
               s=100, alpha=0.6)
    
    # Add diagonal line
    min_val = min(cluster_stats_df['t_accept_d3'].min(), cluster_stats_df['actual_threshold'].min())
    max_val = max(cluster_stats_df['t_accept_d3'].max(), cluster_stats_df['actual_threshold'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Equal')
    
    ax.set_xlabel('Original Idea: median(d3) + 1.5×IQR', fontsize=12)
    ax.set_ylabel('Actual Algorithm Threshold', fontsize=12)
    ax.set_title('Threshold Comparison', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Show which metric is used for threshold
    ax = axes[1]
    x = np.arange(len(cluster_stats_df))
    width = 0.25
    
    ax.bar(x - width, cluster_stats_df['d3_p90'], width, label='d3 P90', alpha=0.7)
    ax.bar(x, cluster_stats_df['pairwise_p90'], width, label='Pairwise P90', alpha=0.7)
    ax.bar(x + width, cluster_stats_df['actual_threshold'], width, label='Actual', alpha=0.7)
    
    ax.set_xlabel('Cluster ID', fontsize=12)
    ax.set_ylabel('Threshold Value', fontsize=12)
    ax.set_title('Threshold Source Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(cluster_stats_df['cluster_id'])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n🔍 Threshold Analysis:")
    print(f"Algorithm: {algorithm}")
    if algorithm == 'hybrid_hdbscan_knn':
        print("  Uses: percentile(exemplar_pairwise_distances, 90)")
    else:
        print("  Uses: percentile(d3_values, 90)")
    print(f"\nYour original idea: median(d3) + 1.5×IQR")
    print(f"\nDifference stats:")
    diff = cluster_stats_df['actual_threshold'] - cluster_stats_df['t_accept_d3']
    print(f"  Mean difference: {diff.mean():.4f}")
    print(f"  Std difference: {diff.std():.4f}")
    print(f"  Min difference: {diff.min():.4f}")
    print(f"  Max difference: {diff.max():.4f}")

## 5. Visualize Cluster Internal Distributions

In [ ]:
# Pick a few clusters to visualize in detail
clusters_to_plot = cluster_stats_df['cluster_id'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, cluster_id in enumerate(clusters_to_plot):
    row = cluster_stats_df[cluster_stats_df['cluster_id'] == cluster_id].iloc[0]
    ax = axes[idx]
    
    # Plot d3 distribution
    d3_values = row['d3_values']
    ax.hist(d3_values, bins=20, alpha=0.6, color='skyblue', edgecolor='black', label='d3 values')
    
    # Add markers for statistics
    ax.axvline(row['d3_median'], color='blue', linestyle='--', linewidth=2, label=f"Median: {row['d3_median']:.3f}")
    ax.axvline(row['d3_p90'], color='purple', linestyle='--', linewidth=2, label=f"P90: {row['d3_p90']:.3f}")
    ax.axvline(row['t_accept_d3'], color='green', linestyle='-', linewidth=2, label=f"Med+1.5×IQR: {row['t_accept_d3']:.3f}")
    
    if 'actual_threshold' in row:
        ax.axvline(row['actual_threshold'], color='red', linestyle='-', linewidth=2, label=f"Actual T: {row['actual_threshold']:.3f}")
    
    ax.set_xlabel('Distance', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f"Cluster {cluster_id} (n={row['n_faces']})", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Debug Specific Merge Decisions

Analyze why specific cluster pairs merged or didn't merge.

In [ ]:
def analyze_merge_decision(cluster_a_id, cluster_b_id, labels, embeddings, stats, config):
    """Analyze a specific merge decision in detail."""
    
    # Get cluster data
    mask_a = labels == cluster_a_id
    mask_b = labels == cluster_b_id
    emb_a = embeddings[mask_a]
    emb_b = embeddings[mask_b]
    
    print(f"\n{'='*80}")
    print(f"MERGE ANALYSIS: Cluster {cluster_a_id} ({len(emb_a)} faces) vs Cluster {cluster_b_id} ({len(emb_b)} faces)")
    print(f"{'='*80}\n")
    
    # Get cluster statistics
    stats_a = cluster_stats_df[cluster_stats_df['cluster_id'] == cluster_a_id].iloc[0]
    stats_b = cluster_stats_df[cluster_stats_df['cluster_id'] == cluster_b_id].iloc[0]
    
    print("📊 Cluster A Statistics:")
    print(f"  d3 median: {stats_a['d3_median']:.4f}")
    print(f"  d3 IQR: {stats_a['d3_iqr']:.4f}")
    print(f"  d3 P90: {stats_a['d3_p90']:.4f}")
    print(f"  Original threshold (med + 1.5×IQR): {stats_a['t_accept_d3']:.4f}")
    if 'actual_threshold' in stats_a:
        print(f"  Actual threshold: {stats_a['actual_threshold']:.4f}")
    
    print("\n📊 Cluster B Statistics:")
    print(f"  d3 median: {stats_b['d3_median']:.4f}")
    print(f"  d3 IQR: {stats_b['d3_iqr']:.4f}")
    print(f"  d3 P90: {stats_b['d3_p90']:.4f}")
    print(f"  Original threshold (med + 1.5×IQR): {stats_b['t_accept_d3']:.4f}")
    if 'actual_threshold' in stats_b:
        print(f"  Actual threshold: {stats_b['actual_threshold']:.4f}")
    
    # Compute cross-cluster distances
    cross_dists = cosine_distance_matrix(emb_a, emb_b)
    min_cross_dist = np.min(cross_dists)
    median_cross_dist = np.median(cross_dists)
    max_cross_dist = np.max(cross_dists)
    
    print("\n🔗 Cross-Cluster Distances:")
    print(f"  Min: {min_cross_dist:.4f}")
    print(f"  Median: {median_cross_dist:.4f}")
    print(f"  Max: {max_cross_dist:.4f}")
    
    # Original idea: Should merge?
    t_a = stats_a['t_accept_d3']
    t_b = stats_b['t_accept_d3']
    
    print("\n💡 Original Idea Analysis:")
    print(f"  Using min(T_A, T_B) = min({t_a:.4f}, {t_b:.4f}) = {min(t_a, t_b):.4f}")
    print(f"  Closest cross-distance: {min_cross_dist:.4f}")
    
    if min_cross_dist <= min(t_a, t_b):
        print("  ✅ Should MERGE (closest distance ≤ threshold)")
    else:
        print("  ❌ Should NOT merge (closest distance > threshold)")
    
    # Count how many pairs are within threshold
    pairs_within_ta = np.sum(cross_dists <= t_a)
    pairs_within_tb = np.sum(cross_dists <= t_b)
    pairs_within_min = np.sum(cross_dists <= min(t_a, t_b))
    
    print(f"\n  Cross-pairs within T_A ({t_a:.4f}): {pairs_within_ta} / {cross_dists.size}")
    print(f"  Cross-pairs within T_B ({t_b:.4f}): {pairs_within_tb} / {cross_dists.size}")
    print(f"  Cross-pairs within min(T_A, T_B): {pairs_within_min} / {cross_dists.size}")
    
    # Find actual decision from debug data
    if 'debug' in stats and 'merge_decisions' in stats['debug']:
        for decision in stats['debug']['merge_decisions']:
            if ((decision['cluster_a'] == cluster_a_id and decision['cluster_b'] == cluster_b_id) or
                (decision['cluster_a'] == cluster_b_id and decision['cluster_b'] == cluster_a_id)):
                print("\n🤖 Actual Algorithm Decision:")
                print(f"  Merged: {decision['merged']}")
                print(f"  Reason: {decision['reason']}")
                print(f"  Threshold used: {decision.get('threshold', 'N/A')}")
                if 'min_distance' in decision:
                    print(f"  Min distance: {decision['min_distance']:.4f}")
                if algorithm == 'hybrid_hdbscan_knn':
                    print(f"  Pairs within T: {decision.get('n_pairs_within', 0)}")
                    print(f"  Required pairs: {config.get('merge_min_pairs', 3)}")
                    print(f"  Exemplars A involved: {decision.get('exemplars_a_involved', 0)}")
                    print(f"  Exemplars B involved: {decision.get('exemplars_b_involved', 0)}")
                    print(f"  Required distinct: {config.get('merge_min_distinct', 2)}")
                else:  # hybrid_closest_face
                    if 'fits_a' in decision:
                        print(f"  Fits A→B: {decision['fits_a']} / {len(emb_a)}")
                        print(f"  Fits B→A: {decision['fits_b']} / {len(emb_b)}")
                        print(f"  Total fits: {decision['n_fits_total']}")
                        print(f"  Required fits: {decision['merge_min_faces']}")
                break
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Cross-distance distribution
    ax = axes[0]
    ax.hist(cross_dists.flatten(), bins=30, alpha=0.6, color='coral', edgecolor='black')
    ax.axvline(t_a, color='blue', linestyle='--', linewidth=2, label=f'T_A (original): {t_a:.3f}')
    ax.axvline(t_b, color='green', linestyle='--', linewidth=2, label=f'T_B (original): {t_b:.3f}')
    if 'actual_threshold' in stats_a:
        ax.axvline(stats_a['actual_threshold'], color='red', linestyle='-', linewidth=2, label=f'Actual T_A: {stats_a["actual_threshold"]:.3f}')
    if 'actual_threshold' in stats_b:
        ax.axvline(stats_b['actual_threshold'], color='orange', linestyle='-', linewidth=2, label=f'Actual T_B: {stats_b["actual_threshold"]:.3f}')
    ax.set_xlabel('Cross-Cluster Distance', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Cross-Distance Distribution\nCluster {cluster_a_id} vs {cluster_b_id}', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Heatmap of cross-distances
    ax = axes[1]
    im = ax.imshow(cross_dists, cmap='viridis', aspect='auto')
    ax.set_xlabel(f'Cluster {cluster_b_id} faces', fontsize=12)
    ax.set_ylabel(f'Cluster {cluster_a_id} faces', fontsize=12)
    ax.set_title('Cross-Distance Heatmap', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Cosine Distance')
    
    plt.tight_layout()
    plt.show()

# Example: Analyze first two clusters
if len(cluster_stats_df) >= 2:
    cluster_a = cluster_stats_df['cluster_id'].iloc[0]
    cluster_b = cluster_stats_df['cluster_id'].iloc[1]
    analyze_merge_decision(cluster_a, cluster_b, labels, embeddings, stats, config)

## 7. Interactive: Pick Your Own Cluster Pair to Debug

In [ ]:
# List all cluster pairs
print("Available clusters:", sorted([c for c in set(labels) if c >= 0]))
print("\nCluster sizes:")
for cid in sorted([c for c in set(labels) if c >= 0]):
    print(f"  Cluster {cid}: {np.sum(labels == cid)} faces")

# Pick two clusters to analyze
cluster_a_id = 0  # Change this
cluster_b_id = 1  # Change this

analyze_merge_decision(cluster_a_id, cluster_b_id, labels, embeddings, stats, config)

## 8. Test Simplified Algorithm: median + k×IQR

Implement and test the original simple idea.

In [ ]:
def simple_threshold_clustering(embeddings, hdbscan_labels, k=3, iqr_multiplier=1.5):
    """
    Simplified clustering using original idea:
    T = median(d3) + iqr_multiplier × IQR(d3)
    
    Returns new labels after merging.
    """
    from collections import defaultdict
    
    labels = hdbscan_labels.copy()
    
    # Compute cluster thresholds
    cluster_thresholds = {}
    cluster_embeddings = {}
    
    for cluster_id in set(labels):
        if cluster_id == -1:
            continue
        
        mask = labels == cluster_id
        cluster_emb = embeddings[mask]
        cluster_embeddings[cluster_id] = cluster_emb
        
        if len(cluster_emb) < 2:
            cluster_thresholds[cluster_id] = 0.3  # Default
            continue
        
        # Compute d3 values
        dist_matrix = cosine_distance_matrix(cluster_emb)
        k_actual = min(k, len(cluster_emb) - 1)
        d3_values = []
        for i in range(len(cluster_emb)):
            sorted_dists = np.sort(dist_matrix[i])[1:k_actual + 1]
            d3 = sorted_dists[-1] if len(sorted_dists) > 0 else 0
            d3_values.append(d3)
        d3_values = np.array(d3_values)
        
        # Simple threshold: median + k×IQR
        median = np.median(d3_values)
        q1, q3 = np.percentile(d3_values, [25, 75])
        iqr = q3 - q1
        threshold = median + iqr_multiplier * iqr
        cluster_thresholds[cluster_id] = threshold
    
    print(f"\nComputed thresholds for {len(cluster_thresholds)} clusters")
    
    # Merge clusters
    cluster_ids = sorted(cluster_thresholds.keys())
    parent = {c: c for c in cluster_ids}
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py
            return True
        return False
    
    n_merges = 0
    merge_log = []
    
    for i, c1 in enumerate(cluster_ids):
        for c2 in cluster_ids[i + 1:]:
            if find(c1) == find(c2):
                continue
            
            # Compute cross-cluster distances
            cross_dists = cosine_distance_matrix(
                cluster_embeddings[c1],
                cluster_embeddings[c2]
            )
            min_dist = np.min(cross_dists)
            
            # Simple rule: merge if closest distance ≤ min(T_A, T_B)
            threshold = min(cluster_thresholds[c1], cluster_thresholds[c2])
            
            if min_dist <= threshold:
                if union(c1, c2):
                    n_merges += 1
                    merge_log.append({
                        'cluster_a': c1,
                        'cluster_b': c2,
                        'min_dist': min_dist,
                        'threshold': threshold,
                        't_a': cluster_thresholds[c1],
                        't_b': cluster_thresholds[c2],
                    })
    
    # Apply merges
    if n_merges > 0:
        label_mapping = {}
        for c in cluster_ids:
            root = find(c)
            if root not in label_mapping:
                label_mapping[root] = len(label_mapping)
        
        for i, lbl in enumerate(labels):
            if lbl >= 0 and lbl in parent:
                labels[i] = label_mapping[find(lbl)]
    
    print(f"Merged {n_merges} cluster pairs")
    
    # Print merge details
    if merge_log:
        print("\nMerge Log:")
        for m in merge_log:
            print(f"  {m['cluster_a']} + {m['cluster_b']}: "
                  f"min_dist={m['min_dist']:.4f} ≤ threshold={m['threshold']:.4f} "
                  f"(T_A={m['t_a']:.4f}, T_B={m['t_b']:.4f})")
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)
    
    return labels, {
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'n_merges': n_merges,
        'cluster_thresholds': cluster_thresholds,
        'merge_log': merge_log,
    }

# Test simplified algorithm
print("\n" + "="*80)
print("TESTING SIMPLIFIED ALGORITHM: median(d3) + 1.5×IQR")
print("="*80)

# Start from HDBSCAN labels
hdbscan_labels = stats['hdbscan']

simple_labels, simple_stats = simple_threshold_clustering(
    embeddings, 
    labels,  # Start from current labels or use hdbscan_labels
    k=3, 
    iqr_multiplier=1.5
)

print(f"\nSimplified Algorithm Results:")
print(f"  Clusters: {simple_stats['n_clusters']}")
print(f"  Noise: {simple_stats['n_noise']}")

# Compare with original
print(f"\nComparison:")
print(f"  Original: {stats['n_clusters']} clusters, {stats['n_noise']} noise")
print(f"  Simplified: {simple_stats['n_clusters']} clusters, {simple_stats['n_noise']} noise")

## 9. Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\n📌 Key Findings:")
print("\n1. THRESHOLD COMPUTATION:")
if algorithm == 'hybrid_hdbscan_knn':
    print("   - Current: Uses percentile(exemplar_pairwise_distances, 90)")
    print("   - Original idea: median(d3) + 1.5×IQR")
    print("   ⚠️  These are DIFFERENT metrics!")
else:
    print("   - Current: Uses percentile(d3_values, 90)")
    print("   - Original idea: median(d3) + 1.5×IQR")
    print("   ⚠️  Percentile vs median+IQR can give different values")

print("\n2. MERGE CRITERIA:")
if algorithm == 'hybrid_hdbscan_knn':
    print("   - Current: Requires ≥3 exemplar pairs within T + ≥2 distinct exemplars")
    print("   - Original idea: Simple distance threshold check")
    print("   ⚠️  Multi-condition check is more restrictive")
else:
    print("   - Current: Counts faces that 'fit' based on d3_cross with 1.5× multiplier")
    print("   - Original idea: Simple closest distance ≤ threshold")
    print("   ⚠️  d3_cross is k-th nearest, not just closest")

print("\n3. COMPLEXITY:")
print("   - Current implementation has many moving parts")
print("   - Hard to debug and understand decisions")
print("   - Your original idea is much simpler and interpretable")

print("\n💡 Recommendations:")
print("\n1. Try simplified algorithm (median + k×IQR) - see Section 8")
print("2. If you need the current complexity, add logging for each decision")
print("3. Consider implementing split logic: median + 2.5×IQR threshold")
print("4. Visualize in UMAP to validate clustering makes sense")

print("\n" + "="*80)

## 10. Next Steps

**To debug further:**
1. Load your actual face embeddings (replace synthetic data in Section 1)
2. Pick specific cluster pairs that should merge (Section 7)
3. Analyze why they merged/didn't merge (Section 6)
4. Test simplified algorithm (Section 8)

**To simplify the algorithm:**
- Modify the clusterer to use `median(d3) + k×IQR` instead of percentile
- Remove complex multi-condition merge checks
- Add split logic based on `median(d3) + 2.5×IQR`